# RD01 — Observable regime discovery and signed-return dynamics

**Exploratory regime tournament: interpretable 27-condition grid versus at-most-eight-leaf CART.** Model baselines: unconditional Student-t and direct feature-conditioned Ridge. Report causal state probabilities, mean signed returns, transition matrices, episode durations and 5/15/30/60-minute density scores.

**Important:** The volatility standardizer is CS04-A's *trailing log_rv24h*, **not point-in-time AM-T-HAR-Blend forecasts**. A fair AM-T density comparison is deferred to RD03. This notebook does not change AM-T, HSMMs, collectors, or existing research outputs. Research protocol: `discussions/Regime_Discovery/001_chatgpt_observable_regime_discovery_preregistration.md`.

The real-data run requires the local `CS04-A` feature store. `RD01_SYNTHETIC=1` runs a synthetic smoke test only.

In [1]:
import os,sys,subprocess,json,time,tempfile,gc
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Open JupyterLab in repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import observable_regime_discovery as rd
cfg=rd.Spec()
SYNTHETIC=os.getenv('RD01_SYNTHETIC','0')=='1'
if SYNTHETIC:
    temp=tempfile.TemporaryDirectory(prefix='rd01-')
    OUT=Path(temp.name)/'01'
else:
    data_root=os.getenv('DERIBIT_DATA_ROOT')
    if not data_root:raise RuntimeError('Set DERIBIT_DATA_ROOT in your .env')
    STORE=Path(data_root)/'research'/'cs04a_feature_store'
    OUT=ROOT/'notebooks'/'Regime_Discovery'/'01'
OUT.mkdir(parents=True,exist_ok=True)
print('RD01',rd.VERSION,'synthetic',SYNTHETIC,'output:',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
r=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_observable_regime_discovery.py')],cwd=ROOT,env=env,text=True,capture_output=True)
print(r.stdout[-1500:]);print(r.stderr[-500:])
assert r.returncode==0,'RD01 tests failed; historical evaluation blocked' 

RD01 RD01-0.1.0 synthetic False output: D:\Projects\deribit_market_data\notebooks\Regime_Discovery\01
...........                                                              [100%]
11 passed in 8.84s




In [2]:
datasets={};excluded={}
if SYNTHETIC:
    sys.path.insert(0,str(ROOT/'tests'))
    from test_observable_regime_discovery import synthetic_panel
    datasets={'BTC':synthetic_panel(days=94)}
else:
    cols=['state_time','return_5m','feature_ready','log_rv24h',*rd.BASE_COLS,*rd.TREE_COLS,
          'ret_5m','ret_15m','ret_30m']
    cols=list(dict.fromkeys(cols))
    for asset in cfg.assets:
        path=STORE/f'{asset}_features_and_forward_magnitude.parquet'
        if not path.exists():
            excluded[asset]=f'CS04-A feature store missing: {path}'
            continue
        try:
            frame=pd.read_parquet(path,columns=cols)
            datasets[asset]=frame
            print(asset,'loaded',len(frame),'rows')
        except Exception as exc:
            excluded[asset]=repr(exc)
assert datasets,f'No usable RD01 inputs: {excluded}'
if not SYNTHETIC and not {'BTC','ETH'}.issubset(datasets):
    print('WARNING: BTC or ETH unavailable; primary cross-asset inference cannot be made')

BTC loaded 221761 rows
ETH loaded 221761 rows
SOL loaded 221761 rows
XRP loaded 221761 rows


In [3]:
results={};coverage=[]
for asset,frame in datasets.items():
    start=time.perf_counter()
    try:
        result=rd.scan_asset(frame,cfg,asset)
        results[asset]=result
        coverage.append({'asset':asset,'n_complete_days':result['n_days'],
                         'n_feature_origins':result['n_rows'],'n_eval_scores':len(result['scores']),
                         'n_state_rows':len(result['regimes']),
                         'n_episodes_local_only':len(result['episodes']),
                         'elapsed_seconds':round(time.perf_counter()-start,1)})
        print(asset,'completed:',len(result['scores']),'model-horizon-fold score rows;',
              result['n_days'],'full days; seconds',round(time.perf_counter()-start,1))
    except Exception as exc:
        excluded[asset]=repr(exc)
        if not SYNTHETIC:raise
    finally:
        del frame
        gc.collect()
assert results,'No assets scanned'
display(pd.DataFrame(coverage))

BTC completed: 48 model-horizon-fold score rows; 736 full days; seconds 26.1
ETH completed: 48 model-horizon-fold score rows; 736 full days; seconds 30.2
SOL completed: 48 model-horizon-fold score rows; 736 full days; seconds 28.7
XRP completed: 48 model-horizon-fold score rows; 736 full days; seconds 28.7


,asset,n_complete_days,n_feature_origins,n_eval_scores,n_state_rows,n_episodes_local_only,elapsed_seconds
0,BTC,736,211968,48,400,55144,26.1
1,ETH,736,211968,48,412,61687,30.2
2,SOL,736,211968,48,416,60189,28.7
3,XRP,736,211968,48,408,58999,28.7


In [4]:
out=rd.consolidate(results,cfg)
out['manifest']['synthetic']=SYNTHETIC
out['manifest']['excluded']=excluded
out['manifest']['n_asset_folds']=len(out['scores'].groupby(['asset','fold']))
print('Average descriptive NLL / conditional UP-DOWN differences (negative = improvement):')
display(out['comparison_summary'].sort_values(['horizon_min','delta_nll'])[['method','horizon_min','n_asset_folds','delta_nll','delta_binary_logloss']])
print('Example state calibrations: predicted UP and observed UP, on held-out nonzero returns')
display(out['state_forecasts'].query('horizon_min==30 and diagnostic_sufficient').sort_values('n',ascending=False).head(20)[
    ['asset','fold','method','regime','n','n_days','pred_up','observed_up_nonzero','nll_delta_vs_baseline']])
print('State persistence relative to within-day iid occupancy (descriptive only):')
display(out['persistence'][['asset','fold','method','observed_stay','within_day_iid_expected_stay','excess_stay_vs_iid']])
print('NOTE: probabilities and states are historical, not live or confirmation-quality.')

Average descriptive NLL / conditional UP-DOWN differences (negative = improvement):


,method,horizon_min,n_asset_folds,delta_nll,delta_binary_logloss
4,direct_ridge,5,12,-0.000855,-0.000712
12,tree_regime,5,12,-0.000253,-0.000264
8,rule_regime,5,12,-0.000203,-0.000160
0,baseline,5,12,0.000000,0.000000
5,direct_ridge,15,12,-0.000396,-0.000391
9,rule_regime,15,12,-0.000192,-0.000224
13,tree_regime,15,12,-0.000165,-0.000155
1,baseline,15,12,0.000000,0.000000
2,baseline,30,12,0.000000,0.000000
14,tree_regime,30,12,0.000007,0.000015


Example state calibrations: predicted UP and observed UP, on held-out nonzero returns


,asset,fold,method,regime,n,n_days,pred_up,observed_up_nonzero,nll_delta_vs_baseline
94,BTC,1,tree_regime,T005,10390,47,0.511987,0.496728,0.000076
1464,XRP,2,tree_regime,T011,8513,47,0.505509,0.486080,-0.000258
776,ETH,3,tree_regime,T008,8089,47,0.502456,0.502163,0.000157
1597,XRP,3,tree_regime,T007,6733,47,0.495058,0.480172,-0.000345
1189,SOL,3,tree_regime,T007,6601,47,0.495251,0.510983,-0.000015
1324,XRP,1,tree_regime,T006,5772,47,0.508064,0.506757,-0.000164
231,BTC,2,tree_regime,T010,5394,47,0.506478,0.501483,-0.000162
640,ETH,2,tree_regime,T010,5360,47,0.519164,0.506716,0.000453
363,BTC,3,tree_regime,T006,5255,47,0.496129,0.487345,-0.000313
1053,SOL,2,tree_regime,T011,5078,47,0.507505,0.503545,-0.000217


State persistence relative to within-day iid occupancy (descriptive only):


,asset,fold,method,observed_stay,within_day_iid_expected_stay,excess_stay_vs_iid
0,BTC,1,rule_regime,0.124723,0.088252,0.036471
1,BTC,1,tree_regime,0.659894,0.631707,0.028187
2,BTC,2,rule_regime,0.111119,0.076488,0.034631
3,BTC,2,tree_regime,0.530624,0.330215,0.200410
4,BTC,3,rule_regime,0.112375,0.085915,0.026460
5,BTC,3,tree_regime,0.387588,0.258912,0.128676
6,ETH,1,rule_regime,0.106399,0.082057,0.024342
7,ETH,1,tree_regime,0.161667,0.190726,-0.029059
8,ETH,2,rule_regime,0.099889,0.073642,0.026247
9,ETH,2,tree_regime,0.333727,0.248564,0.085163


NOTE: probabilities and states are historical, not live or confirmation-quality.


In [5]:
files={
    '01_scores.csv':'scores',
    '01_comparison_summary.csv':'comparison_summary',
    '01_paired_bootstrap.csv':'paired_bootstrap',
    '01_regime_characteristics.csv':'regimes',
    '01_state_forecasts.csv':'state_forecasts',
    '01_transition_matrix.csv':'transitions',
    '01_episode_summary.csv':'episode_summary',
    '01_exit_hazards.csv':'hazards',
    '01_persistence.csv':'persistence',
    '01_rule_definitions.csv':'rule_definitions',
    '01_tree_definitions.csv':'tree_definitions',
}
for filename,key in files.items():out[key].to_csv(OUT/filename,index=False)
pd.DataFrame(coverage).to_csv(OUT/'01_coverage.csv',index=False)
(OUT/'01_manifest.json').write_text(json.dumps(out['manifest'],indent=2,default=str),encoding='utf-8')
assert len(list(OUT.iterdir()))==13,'Expected exactly 12 aggregate CSV + one JSON manifest'
print('Wrote 13 aggregate outputs to',OUT)
print('Do NOT push the local CS04-A Parquet file or individual episode/origin rows.')

Wrote 13 aggregate outputs to D:\Projects\deribit_market_data\notebooks\Regime_Discovery\01
Do NOT push the local CS04-A Parquet file or individual episode/origin rows.


## Interpretation constraints

- A partition that has different historical return frequencies is **not automatically a persistent market regime**. Check occupancy, transitions, censored durations, stability and nulls.
- The training tree splits were chosen using standardized **30-minute** future return from its STRUCTURE training subset, while its probability/density parameters were calibrated on a later, disjoint training subset. No validation labels enter training.
- State IDs such as `T004` are **fold-local**, not structural identities. Compare their actual split rules, not just numeric IDs.
- Binary UP/DOWN probabilities exclude exact ZERO targets for scoring; whether a return will be nonzero is unknown when the forecast is made.
- Paired confidence intervals are unadjusted and exploratory; choosing high-probability states from these already-examined periods inflates discovery bias.
- RD01 **does not validate** AM-T-HAR-Blend integration. RD02 compares learned/latent state methods, RD03 evaluates full matched conditional return density, and prospective confirmation follows a frozen model specification.